# nb_05_silver_coverages

**Layer**: Silver | **Source**: `lh_bronze_insurance.coverages_raw` | **Target**: `lh_silver_insurance.coverages`

**Data Quality Rules**:
- Cast `coverage_limit`, `deductible_amount` to DoubleType
- Require non-null: `coverage_id`, `policy_id`, `coverage_type`
- Deduplicate on `coverage_id` (keep latest `_ingested_at`)
- Standardize `coverage_type` to lowercase
- Rejects → `coverages_quarantine`

**Idempotency**: Uses `overwrite` mode.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, current_timestamp, lit, row_number, when, coalesce, lower
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType

try:
    spark
except NameError:
    spark = SparkSession.builder.appName("nb_05_silver_coverages").getOrCreate()

print("nb_05_silver_coverages - Silver Layer Transformation")

In [ ]:
# ============================================================
# Step 1: Read from Bronze
# ============================================================
df_bronze = spark.table("coverages_raw")
bronze_count = df_bronze.count()
print(f"Bronze rows read: {bronze_count:,}")
df_bronze.printSchema()

In [ ]:
# ============================================================
# Step 2: Cast types & clean
# ============================================================
df_typed = df_bronze \
    .withColumn("coverage_id", trim(col("coverage_id"))) \
    .withColumn("policy_id", trim(col("policy_id"))) \
    .withColumn("coverage_type", lower(trim(col("coverage_type")))) \
    .withColumn("coverage_limit", col("coverage_limit").cast(DoubleType())) \
    .withColumn("deductible_amount", col("deductible_amount").cast(DoubleType()))

print("Type casting complete.")

In [ ]:
# ============================================================
# Step 3: Validate & route rejects
# ============================================================
REQUIRED_FIELDS = ["coverage_id", "policy_id", "coverage_type"]

rejection_conditions = [
    when((col(f).isNull()) | (col(f) == ""), lit(f"null_{f}"))
    for f in REQUIRED_FIELDS
]

df_validated = df_typed.withColumn("_rejection_reason", coalesce(*rejection_conditions))

df_valid = df_validated.filter(col("_rejection_reason").isNull()).drop("_rejection_reason")
df_rejected = df_validated.filter(col("_rejection_reason").isNotNull())

valid_count = df_valid.count()
rejected_count = df_rejected.count()
print(f"Valid: {valid_count:,} | Rejected: {rejected_count:,}")

In [ ]:
# ============================================================
# Step 4: Deduplicate on coverage_id
# ============================================================
window_spec = Window.partitionBy("coverage_id").orderBy(col("_ingested_at").desc())

df_deduped = df_valid \
    .withColumn("_row_num", row_number().over(window_spec)) \
    .filter(col("_row_num") == 1) \
    .drop("_row_num")

deduped_count = df_deduped.count()
dupes_removed = valid_count - deduped_count
print(f"After dedup: {deduped_count:,} ({dupes_removed} duplicates removed)")

In [ ]:
# ============================================================
# Step 5: Write Silver & Quarantine
# ============================================================
silver_columns = ["coverage_id", "policy_id", "coverage_type",
                  "coverage_limit", "deductible_amount"]

df_silver = df_deduped.select(silver_columns) \
    .withColumn("_processed_at", current_timestamp())

df_silver.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("coverages")
print(f"✓ {deduped_count:,} rows → coverages")

if rejected_count > 0:
    df_rejected.withColumn("_quarantined_at", current_timestamp()) \
        .write.format("delta").mode("overwrite") \
        .option("overwriteSchema", "true").saveAsTable("coverages_quarantine")
    print(f"✓ {rejected_count:,} rows → coverages_quarantine")

In [ ]:
# ============================================================
# Validation
# ============================================================
print("\nSILVER COVERAGES - SUMMARY")
print("=" * 50)
print(f"  Bronze input:       {bronze_count:>8,}")
print(f"  Rejected:           {rejected_count:>8,}")
print(f"  Duplicates removed: {dupes_removed:>8,}")
print(f"  Silver output:      {deduped_count:>8,}")
print(f"  Quality rate:       {(deduped_count/bronze_count*100):.1f}%")

assert spark.table("coverages").count() == deduped_count
print("\n✓ Verification passed")
spark.table("coverages").show(5, truncate=25)